In [2]:
import pandas as pd
import sqlite3

# load the cleaned dataframes
df_water_production = pd.read_csv('water_production.csv')
df_water_consumption = pd.read_csv('water_consumption.csv')
df_water_access = pd.read_csv('water_access.csv')

# convert date columns
df_water_production['date'] = pd.to_datetime(df_water_production['date'])
df_water_consumption['date'] = pd.to_datetime(df_water_consumption['date'])
df_water_access['date'] = pd.to_datetime(df_water_access['date'])

# rename value columns
df_water_production = df_water_production.rename(columns={'value': 'water_production'})
df_water_consumption = df_water_consumption.rename(columns={'value': 'water_consumption'})

# filter to 2003 onwards
df_water_production = df_water_production[df_water_production['date'].dt.year >= 2003]
df_water_consumption = df_water_consumption[df_water_consumption['date'].dt.year >= 2003]
df_water_access = df_water_access[df_water_access['date'].dt.year >= 2003]

# create the SQLite database and load all three tables
conn = sqlite3.connect('malaysia_water.db')

df_water_production.to_sql('water_production', conn, if_exists='replace', index=False)
df_water_consumption.to_sql('water_consumption', conn, if_exists='replace', index=False)
df_water_access.to_sql('water_access', conn, if_exists='replace', index=False)

conn.close()
print("Database is created. All three tables loaded successfully.")

Database is created. All three tables loaded successfully.


In [3]:
import pandas as pd
import sqlite3

df_water_production = pd.read_csv('water_production.csv')
print("production loaded:", df_water_production.shape)

df_water_consumption = pd.read_csv('water_consumption.csv')
print("consumption loaded:", df_water_consumption.shape)

df_water_access = pd.read_csv('water_access.csv')
print("access loaded:", df_water_access.shape)

production loaded: (345, 3)
consumption loaded: (600, 4)
access loaded: (1035, 4)


In [4]:
# convert dates and rename columns
df_water_production['date'] = pd.to_datetime(df_water_production['date'])
df_water_consumption['date'] = pd.to_datetime(df_water_consumption['date'])
df_water_access['date'] = pd.to_datetime(df_water_access['date'])

df_water_production = df_water_production.rename(columns={'value': 'water_production'})
df_water_consumption = df_water_consumption.rename(columns={'value': 'water_consumption'})

# filter to 2003 onwards
df_water_production = df_water_production[df_water_production['date'].dt.year >= 2003]
df_water_consumption = df_water_consumption[df_water_consumption['date'].dt.year >= 2003]
df_water_access = df_water_access[df_water_access['date'].dt.year >= 2003]

print("Dataframes cleaned and filtered")
print("Production:", df_water_production.shape)
print("Consumption:", df_water_consumption.shape)
print("Access:", df_water_access.shape)

Dataframes cleaned and filtered
Production: (300, 3)
Consumption: (600, 4)
Access: (900, 4)


In [5]:
conn = sqlite3.connect('malaysia_water.db')
print("Database connected")

df_water_production.to_sql('water_production', conn, if_exists='replace', index=False)
print("Production table loaded")

df_water_consumption.to_sql('water_consumption', conn, if_exists='replace', index=False)
print("Consumption table loaded")

df_water_access.to_sql('water_access', conn, if_exists='replace', index=False)
print("Access table loaded")

conn.close()
print("Done — malaysia_water.db created")

Database connected
Production table loaded
Consumption table loaded
Access table loaded
Done — malaysia_water.db created


In [6]:
conn = sqlite3.connect('malaysia_water.db')

for table in ['water_production', 'water_consumption', 'water_access']:
    df = pd.read_sql(f"SELECT COUNT(*) as rows FROM {table}", conn)
    print(f"{table}: {df['rows'][0]} rows")

conn.close()

water_production: 300 rows
water_consumption: 600 rows
water_access: 900 rows


In [7]:
import os

db_path = 'malaysia_water.db'
if os.path.exists(db_path):
    size = os.path.getsize(db_path)
    print(f"Database found: {db_path}")
    print(f"Size: {size:,} bytes")
else:
    print("Database not found — need to recreate it")

Database found: malaysia_water.db
Size: 106,496 bytes


In [8]:
import os
os.makedirs('dashboard', exist_ok=True)
conn = sqlite3.connect('malaysia_water.db')

queries = {
    'State Ranking': """
        SELECT state,
            ROUND(AVG(proportion), 2) AS avg_access,
            ROUND(MIN(proportion), 2) AS min_access,
            ROUND(MAX(proportion), 2) AS max_access
        FROM water_access
        WHERE strata = 'overall' AND state != 'Malaysia'
        GROUP BY state ORDER BY avg_access ASC""",

    'Production and Consumption': """
        SELECT strftime('%Y', date) AS year,
            SUM(water_production) AS total_production,
            (SELECT SUM(wc.water_consumption) FROM water_consumption wc 
             WHERE strftime('%Y', wc.date) = strftime('%Y', wp.date)) AS total_consumption,
            SUM(water_production) - 
            (SELECT SUM(wc.water_consumption) FROM water_consumption wc 
             WHERE strftime('%Y', wc.date) = strftime('%Y', wp.date)) AS gap,
            ROUND((SELECT SUM(wc.water_consumption) FROM water_consumption wc 
             WHERE strftime('%Y', wc.date) = strftime('%Y', wp.date)) * 100.0 
             / SUM(water_production), 2) AS consumption_pct
        FROM water_production wp
        GROUP BY year ORDER BY year ASC""",

    'Year-over-Year Access': """
        SELECT strftime('%Y', date) AS year,
            ROUND(AVG(proportion), 2) AS national_access,
            ROUND(AVG(proportion) - LAG(AVG(proportion)) 
                OVER (ORDER BY strftime('%Y', date)), 2) AS yoy_change
        FROM water_access
        WHERE state = 'Malaysia' AND strata = 'overall'
        GROUP BY year ORDER BY year ASC""",

    'Sector Consumption': """
        SELECT strftime('%Y', date) AS year,
            ROUND(SUM(CASE WHEN sector = 'domestic' 
                THEN water_consumption ELSE 0 END), 2) AS domestic_consumption,
            ROUND(SUM(CASE WHEN sector = 'nondomestic' 
                THEN water_consumption ELSE 0 END), 2) AS nondomestic_consumption,
            ROUND(SUM(CASE WHEN sector = 'domestic' 
                THEN water_consumption ELSE 0 END) * 100.0 
                / SUM(water_consumption), 2) AS domestic_pct,
            ROUND(SUM(CASE WHEN sector = 'nondomestic' 
                THEN water_consumption ELSE 0 END) * 100.0 
                / SUM(water_consumption), 2) AS nondomestic_pct
        FROM water_consumption
        GROUP BY year ORDER BY year ASC""",

    'Underserved States': """
        SELECT state,
            ROUND(AVG(proportion), 2) AS avg_access,
            ROUND(AVG(proportion) - (
                SELECT AVG(proportion) FROM water_access 
                WHERE state = 'Malaysia' AND strata = 'overall'
            ), 2) AS gap_from_national_avg
        FROM water_access
        WHERE strata = 'overall' AND state != 'Malaysia'
        GROUP BY state
        HAVING AVG(proportion) < (
            SELECT AVG(proportion) FROM water_access 
            WHERE state = 'Malaysia' AND strata = 'overall'
        )
        ORDER BY avg_access ASC""",

    'Urban and Rural Gap': """
        SELECT state,
            ROUND(AVG(CASE WHEN strata = 'urban' 
                THEN proportion END), 2) AS avg_urban_access,
            ROUND(AVG(CASE WHEN strata = 'rural' 
                THEN proportion END), 2) AS avg_rural_access,
            ROUND(AVG(CASE WHEN strata = 'urban' 
                THEN proportion END) - 
            AVG(CASE WHEN strata = 'rural' 
                THEN proportion END), 2) AS urban_rural_gap
        FROM water_access
        WHERE state != 'Malaysia'
        GROUP BY state ORDER BY urban_rural_gap DESC"""
}

for filename, query in queries.items():
    df = pd.read_sql(query, conn)
    df.to_csv(f'dashboard/{filename}.csv', index=False)
    print(f"Exported: {filename}.csv — {len(df)} rows")

conn.close()
print("\nAll six CSVs ready in dashboard folder")

Exported: State Ranking.csv — 14 rows
Exported: Production and Consumption.csv — 20 rows
Exported: Year-over-Year Access.csv — 20 rows
Exported: Sector Consumption.csv — 20 rows
Exported: Underserved States.csv — 4 rows
Exported: Urban and Rural Gap.csv — 14 rows

All six CSVs ready in dashboard folder
